# Period moves — outrights + curves/flies/spreads

Introduce **START** y **END**. El notebook calcula el cambio en ese intervalo y la **std de movimientos del mismo horizonte** (si el periodo ≈ 1 día → std de cambios diarios; ≈ 1 semana → semanales; etc., vía `# barras` del panel).

`REFRESH = True` (default) actualiza `sofr_live` hasta ahora antes de calcular. `DAILY = True` usa el mismo `bdh` diario que `run_pull_data`.

### Formato de fechas (elige uno)
```
10/15/26 09:00
10/15/26 9am
2026-10-15 09:00
10/15/2026 09:00
```

Listas editables en `curve_config.py`: `MOVE_OUTRIGHTS`, `MOVE_STRUCTURES`.


In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "lib" / "period_moves.py").exists() else _DZ
if not (REPO / "lib" / "period_moves.py").exists():
    raise FileNotFoundError("No lib/period_moves.py — git pull origin main")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())


In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Fechas + force-reload ────────────────────────────────────────────────────
START = "10/15/26 09:00"
END   = ""            # "" → ahora

REFRESH = True
DAILY = True

import importlib, subprocess
from pathlib import Path
import pandas as pd
import curve_config
import lib.data as data
import lib.period_moves as period_moves

importlib.reload(curve_config)
importlib.reload(data)
importlib.reload(period_moves)

outrights_cfg = getattr(curve_config, "MOVE_OUTRIGHTS", ["2", "5", "10", "30"])
structures_cfg = getattr(
    curve_config, "MOVE_STRUCTURES",
    ["2s5s", "5s10s", "10s30s", "2s5s10s", "5s10s30s", "USSFCT05", "USSFCT10", "USSFCT30"],
)

end_arg = END.strip() if isinstance(END, str) and END.strip() else pd.Timestamp.now().strftime("%m/%d/%y %H:%M")

store = data.resolve_live_store_path()
print("REPO cwd :", Path.cwd().resolve())
print("store    :", store, " exists=", store.exists())
try:
    print("git HEAD :", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
except Exception as e:
    print("git HEAD : (unavailable)", e)
print("has update_live_store_to_now:", hasattr(data, "update_live_store_to_now"))
print("has tickers_for_names      :", hasattr(data, "tickers_for_names"))
print("START:", START, "→", period_moves.parse_ts(START))
print("END  :", repr(END), "→", period_moves.parse_ts(end_arg))
if store.exists():
    raw = data.load_live_store(store)
    print("store asof BEFORE any refresh:", raw.index[-1], " n=", len(raw))



In [ ]:
# ── Refresh + tablas ─────────────────────────────────────────────────────────
outrights, structures, meta, panel = period_moves.run_period_moves(
    START,
    end_arg,
    outrights=outrights_cfg,
    structures=structures_cfg,
    refresh=REFRESH,
    prefer_daily=DAILY,
)
print("horizon:", meta["horizon"])
print("bars used for std:", meta["n_bars"], "@", meta["bar_minutes"], "min")
print("panel asof:", panel.index[-1])
store = data.resolve_live_store_path()
raw = data.load_live_store(store)
print("store asof AFTER:", raw.index[-1], " n=", len(raw))
if raw.index[-1] < pd.Timestamp.now() - pd.Timedelta(days=5):
    raise RuntimeError(
        "sofr_live asof is still stale (%s). "
        "Scroll up: did BBG print [bdh] bars=… or 'keeping existing store'?"
        % raw.index[-1]
    )



In [ ]:
# ── Tabla 1: OUTRIGHTS (2y/5y/10y/30y) en bp ─────────────────────────────────
print("OUTRIGHTS — change & std in bp  |  z = change / std(same-horizon moves)")
display(outrights)

In [ ]:
# ── Tabla 2: CURVES / FLIES / SPREADS (bp / BBG units) ───────────────────────
print("STRUCTURES — 2s5s, 5s10s, 10s30s, flies, USSFCT…")
display(structures)

In [ ]:
# Guardar CSV (opcional)
out_dir = Path("graphs/period_moves")
out_dir.mkdir(parents=True, exist_ok=True)
outrights.to_csv(out_dir / "moves_outrights.csv", index=False)
structures.to_csv(out_dir / "moves_structures.csv", index=False)
print("wrote", out_dir.resolve())